# IoT-Enriched Sensor Playout Case Study

This notebook evaluates the discovered process model as a **generative** model. It uses the `data/IoT-Enriched Event Logs` SensorStream data, discovers a sensor-only Petri-net model, replays that model, generates new sensor traces from the learned rule alphabet with an SMT-backed sampler, and compares real-vs-generated distributions.

The goal is not to reproduce the original samples exactly. The goal is to check whether model playout can recreate similar **sensor value** and **case-level area** distributions.

In [ ]:
from pathlib import Path
import importlib
import sys

repo = Path.cwd()
for candidate in [repo, *repo.parents]:
    if (candidate / "src").exists():
        repo = candidate
        break
sys.path.insert(0, str(repo))

dataset_dir = repo / "data" / "IoT-Enriched Event Logs"
sys.path.insert(0, str(dataset_dir))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sensorstream_dataset import (
    find_data_dir,
    load_sensor_cases,
    parse_dataset,
    write_split_outputs,
)
from src.evaluation import stack_case_arrays
from src.pipeline import PipelineConfig, run_pipeline
import src.playout as playout_module

# Jupyter keeps imported modules alive between cell runs. Reload this module so
# the notebook picks up local generator changes without a kernel restart.
playout_module = importlib.reload(playout_module)
PlayoutConfig = playout_module.PlayoutConfig
case_area_distribution = playout_module.case_area_distribution
compare_case_area_distributions = playout_module.compare_case_area_distributions
compare_sensor_value_distributions = playout_module.compare_sensor_value_distributions
playout_sensor_log = playout_module.playout_sensor_log
wasserstein_1d = playout_module.wasserstein_1d

## 1. Load a Small IoT-Enriched Sensor Sample

The splitter is reused from the dataset folder. It separates `MainProcess.xes`, linked low-level subprocess XES files, and flattened SensorStream points. The high-level log is kept as baseline metadata for the sampled sensor-bearing subprocess invocations, but the discovery step still receives only sensor values.

In [ ]:
split_dir = dataset_dir / "derived" / "split"
required = [split_dir / "sensor_matrix.csv", split_dir / "high_level_events.csv"]
if not all(path.exists() for path in required):
    data_dir = find_data_dir(dataset_dir)
    main_events, invocations, subprocess_events, stream_points = parse_dataset(data_dir)
    write_split_outputs(split_dir, main_events, invocations, subprocess_events, stream_points)

high_level_events = pd.read_csv(split_dir / "high_level_events.csv")
cases, case_metadata, sensor_cols = load_sensor_cases(split_dir / "sensor_matrix.csv")

# Keep the sample small and reproducible. The current dataset has 10 non-empty linked subprocess streams.
N_CASES = min(10, len(cases))
cases = cases[:N_CASES]
case_metadata = case_metadata[:N_CASES]

arrays = [case.to_numpy(dtype=float) for case in cases]
real_data, real_case_boundaries = stack_case_arrays(arrays)

print(f"High-level linked invocations: {len(high_level_events)}")
print(f"Sensor-bearing invocations sampled: {len(cases)}")
print(f"Samples x sensors: {real_data.shape}")
sample_table = pd.DataFrame(case_metadata)[["order", "baseline_activity", "n_samples"]]
sample_table

## 2. Discover a Sensor-Only Process Model

The configuration mirrors the stronger setting from the IoT-enriched DFG evaluation: case-local segmentation, robust feature scaling, derivative features, and depth-2 signatures. No high-level labels are used during discovery.

In [ ]:
discovery = run_pipeline(PipelineConfig(
    data=real_data,
    penalty=1.5,
    min_segment_size=2,
    segment_by_case=True,
    merge_short_segments=True,
    var_names=sensor_cols,
    case_boundaries=real_case_boundaries,
    signature_depth=2,
    include_derivative_features=True,
    activity_abstraction="clustered_interval",
    n_activity_clusters=18,
    max_profile_features=128,
    profile_scaling="robust",
    rule_margin=0.1,
    activity_label_style="generic",
))

print(f"Segments: {len(discovery.segment_labels)}")
print(f"Activities: {discovery.n_classes}")
print(f"Trace variants: {len({tuple(trace) for trace in discovery.traces})}")
print(f"Petri net: {len(discovery.net.transitions)} transitions, {len(discovery.net.places)} places")

## 3. Replay the Model and Generate New Sensor Logs

Each replayed visible activity uses Z3 to solve a feasible feature vector for its learned interval rule. The SMT model enforces the selected rule intervals plus generic path constraints such as `min <= start/end <= max` and `sig(sensor) = end - start`. The path reconstruction then uses those solved features, so this is stricter than independent interval sampling, but still not a full inverse-signature solver for the entire time series.

In [ ]:
generated = playout_sensor_log(
    discovery,
    PlayoutConfig(
        n_cases=100,
        min_segment_length=490,
        max_segment_length=500,
        noise_scale=0.03,
        random_state=21,
        feature_sampler="smt",
        smt_timeout_ms=500,
    ),
)

print(f"Generated samples x sensors: {generated.data.shape}")
print(f"Generated trace variants: {len({tuple(trace) for trace in generated.traces})}")
generated.traces[:5]

## 4. Compare Real vs Generated Distributions

The plot below follows the same idea as the example image: all comparisons are in one figure, with transparent overlaid histograms and outline lines. The rows compare point values, per-case mean values, and absolute case-level area-under-curve values. The last row is intentionally duration-sensitive, so it exposes when Petri-net replay creates longer or shorter cases than the real sensor sample.

In [ ]:
value_comparison = compare_sensor_value_distributions(
    discovery.preprocessed_data,
    generated.data,
    discovery.sensor_names,
)
area_comparison = compare_case_area_distributions(
    discovery.preprocessed_data,
    real_case_boundaries,
    generated.data,
    generated.case_boundaries,
    discovery.sensor_names,
)

real_var = pd.Series(discovery.preprocessed_data.var(axis=0), index=discovery.sensor_names)
selected_sensors = real_var.sort_values(ascending=False).head(4).index.tolist()
selected_sensors

In [ ]:
def short_sensor_name(name: str, width: int = 32) -> str:
    text = name.replace("__", " / ").replace("_", " ")
    if len(text) <= width:
        return text
    return text[:width - 3] + "..."


def overlay_hist(ax, real, generated_values, title, xlabel, distance):
    values = np.concatenate([real, generated_values])
    if np.allclose(values.min(), values.max()):
        bins = 5
    else:
        bins = np.linspace(values.min(), values.max(), 28)
    real_color = "#4C78A8"
    gen_color = "#F58518"
    ax.hist(real, bins=bins, density=True, alpha=0.28, color=real_color, label="Real")
    ax.hist(generated_values, bins=bins, density=True, alpha=0.28, color=gen_color, label="Generated")
    ax.hist(real, bins=bins, density=True, histtype="step", linewidth=2.0, color=real_color)
    ax.hist(generated_values, bins=bins, density=True, histtype="step", linewidth=2.0, color=gen_color)
    ax.set_title(title, fontsize=11.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.text(
        0.98,
        0.92,
        f"W1={distance:.3f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=10,
        bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#d7d7d7"},
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


real_areas = case_area_distribution(discovery.preprocessed_data, real_case_boundaries, discovery.sensor_names)
generated_areas = case_area_distribution(generated.data, generated.case_boundaries, generated.sensor_names)
mean_comparison = pd.DataFrame([
    {
        "sensor": sensor,
        "mean_wasserstein_1d": wasserstein_1d(
            real_areas.loc[real_areas["sensor"] == sensor, "mean"].to_numpy(),
            generated_areas.loc[generated_areas["sensor"] == sensor, "mean"].to_numpy(),
        ),
    }
    for sensor in discovery.sensor_names
])

fig, axes = plt.subplots(3, len(selected_sensors), figsize=(4.8 * len(selected_sensors), 11.8), constrained_layout=True)
for col, sensor in enumerate(selected_sensors):
    sensor_idx = discovery.sensor_names.index(sensor)
    sensor_title = f"S{col + 1}: {short_sensor_name(sensor)}"
    value_distance = float(value_comparison.loc[value_comparison["sensor"] == sensor, "wasserstein_1d"].iloc[0])
    mean_distance = float(mean_comparison.loc[mean_comparison["sensor"] == sensor, "mean_wasserstein_1d"].iloc[0])
    area_distance = float(area_comparison.loc[area_comparison["sensor"] == sensor, "area_wasserstein_1d"].iloc[0])
    overlay_hist(
        axes[0, col],
        discovery.preprocessed_data[:, sensor_idx],
        generated.data[:, sensor_idx],
        sensor_title,
        "Normalized sensor value",
        value_distance,
    )
    overlay_hist(
        axes[1, col],
        real_areas.loc[real_areas["sensor"] == sensor, "mean"].to_numpy(),
        generated_areas.loc[generated_areas["sensor"] == sensor, "mean"].to_numpy(),
        "Case mean",
        "Mean normalized value",
        mean_distance,
    )
    overlay_hist(
        axes[2, col],
        real_areas.loc[real_areas["sensor"] == sensor, "area"].to_numpy(),
        generated_areas.loc[generated_areas["sensor"] == sensor, "area"].to_numpy(),
        "Case area",
        "Area under curve",
        area_distance,
    )

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncols=2, frameon=True, bbox_to_anchor=(0.5, 1.025))
fig.suptitle("IoT Sensor Playout: Real vs Generated Distributions", fontsize=18, fontweight="bold", y=1.06)
output_path = Path("tests/case_study/iot_playout_distribution_comparison.png")
fig.savefig(repo / output_path, dpi=220, bbox_inches="tight")
plt.show()
output_path

## 5. Numeric Summary for the Selected Sensors

In [ ]:
value_comparison[value_comparison["sensor"].isin(selected_sensors)].sort_values("wasserstein_1d")

In [ ]:
mean_comparison[mean_comparison["sensor"].isin(selected_sensors)].sort_values("mean_wasserstein_1d")

In [ ]:
area_comparison[area_comparison["sensor"].isin(selected_sensors)].sort_values("area_wasserstein_1d")

## Interpretation

This case study asks a different question than conformance checking. It asks whether the discovered model can generate new sensor logs with similar distributional behavior. Good agreement in the top-row histograms means the rule alphabet preserves point-value behavior. Good agreement in the middle row means generated cases preserve normalized exposure independent of case length. The bottom row is stricter because it includes replay duration; large distances there indicate that the Petri-net control flow may be plausible while the replayed case lengths or rule-based path reconstruction are still too weak for that sensor.